In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Tentukan opsi

<details>
<summary><b>Versi paket</b></summary>

Kode di halaman ini dikembangkan menggunakan persyaratan berikut.
Kami menyarankan untuk menggunakan versi ini atau yang lebih baru.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Kamu bisa menggunakan opsi untuk menyesuaikan primitif Estimator dan Sampler. Bagian ini berfokus pada cara menentukan opsi primitif Qiskit Runtime. Meskipun antarmuka metode `run()` primitif sama di semua implementasi, opsinya tidak. Lihat referensi API yang sesuai untuk informasi tentang opsi [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives#primitives) dan [`qiskit_aer.primitives`](https://qiskit.github.io/qiskit-aer/apidocs/aer_primitives.html).

Catatan tentang menentukan opsi dalam primitif:

- `SamplerV2` dan `EstimatorV2` memiliki kelas opsi terpisah. Kamu bisa melihat opsi yang tersedia dan memperbarui nilai opsi selama atau setelah inisialisasi primitif.
- Gunakan metode `update()` untuk menerapkan perubahan pada atribut `options`.
- Jika kamu tidak menentukan nilai untuk suatu opsi, nilainya diberi nilai khusus `Unset` dan default server akan digunakan.
- Atribut `options` adalah tipe Python `dataclass`. Kamu bisa menggunakan metode bawaan `asdict` untuk mengonversinya menjadi dictionary.

<span id="pass-options"></span>
## Atur opsi primitif
Kamu bisa mengatur opsi saat menginisialisasi primitif, setelah menginisialisasi primitif, atau dalam metode `run()`. Lihat bagian [aturan prioritas](runtime-options-overview#options-precedence) untuk memahami apa yang terjadi ketika opsi yang sama ditentukan di beberapa tempat.

### Inisialisasi primitif
Kamu bisa memasukkan instance kelas opsi atau dictionary saat menginisialisasi primitif, yang kemudian membuat salinan opsi tersebut. Jadi, mengubah dictionary atau instance opsi asli tidak memengaruhi opsi yang dimiliki primitif.

#### Kelas opsi
Saat membuat instance kelas `EstimatorV2` atau `SamplerV2`, kamu bisa memasukkan instance kelas opsi. Opsi tersebut kemudian akan diterapkan saat kamu menggunakan `run()` untuk melakukan perhitungan. Tentukan opsi dalam format ini: `options.option.sub-option.sub-sub-option = choice`. Contohnya: `options.dynamical_decoupling.enable = True`

Contoh:

`SamplerV2` dan `EstimatorV2` memiliki kelas opsi terpisah ([`EstimatorOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options) dan [`SamplerOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options)).

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime.options import EstimatorOptions

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

options = EstimatorOptions(
    resilience_level=2,
    resilience={"zne_mitigation": True, "zne": {"noise_factors": [1, 3, 5]}},
)

# or...
options = EstimatorOptions()
options.resilience_level = 2
options.resilience.zne_mitigation = True
options.resilience.zne.noise_factors = [1, 3, 5]

estimator = Estimator(mode=backend, options=options)

#### Dictionary
Kamu bisa menentukan opsi sebagai dictionary saat menginisialisasi primitif.

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(
    backend,
    options={
        "resilience_level": 2,
        "resilience": {
            "zne_mitigation": True,
            "zne": {"noise_factors": [1, 3, 5]},
        },
    },
)

### Perbarui opsi setelah inisialisasi
Kamu bisa menentukan opsi dalam format ini: `primitive.options.option.sub-option.sub-sub-option = choice` untuk memanfaatkan auto-complete, atau gunakan metode `update()` untuk melakukan pembaruan massal.

Kelas opsi `SamplerV2` dan `EstimatorV2` ([`EstimatorOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options) dan [`SamplerOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options)) tidak perlu diinstansiasi jika kamu mengatur opsi setelah menginisialisasi primitif.

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(mode=backend)

# Setting options after primitive initialization
# This uses auto-complete.
estimator.options.default_shots = 4000
# This does bulk update.
estimator.options.update(
    default_shots=4000, resilience={"zne_mitigation": True}
)

<span id="run-method"></span>
### Metode Run()
Satu-satunya nilai yang bisa kamu masukkan ke `run()` adalah yang didefinisikan dalam antarmuka. Yaitu, `shots` untuk Sampler dan `precision` untuk Estimator. Ini menimpa nilai yang ditetapkan untuk `default_shots` atau `default_precision` untuk run saat ini.

In [4]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)

sampler = Sampler(mode=backend)
# Default shots to use if not specified in run()
sampler.options.default_shots = 500
# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

# Sample two circuits with different numbers of shots.
# 100 shots is used for transpiled1 and 200 for transpiled.
sampler.run([(transpiled1, None, 100), (transpiled2, None, 200)])

<RuntimeJobV2('d5k96cn853es738djikg', 'sampler')>

### Special cases

#### Resilience level (Estimator only)

The resilience level is not actually an option that directly impacts the primitive query, but specifies a base set of curated options to build off of. In general, level 0 turns off all error mitigation, level 1 turns on options for measurement error mitigation, and level 2 turns on options for gate and measurement error mitigation.

Any options you manually specify in addition to the resilience level are applied on top of the base set of options defined by the resilience level. Therefore, in principle, you could set the resilience level to 1, but then turn off measurement mitigation, although this is not advised.

In the following example, setting the resilience level to 0 initially turns off `zne_mitigation`, but `estimator.options.resilience.zne_mitigation = True` overrides the relevant setup from `estimator.options.resilience_level = 0`.

In [5]:
from qiskit_ibm_runtime import EstimatorV2, QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = EstimatorV2(backend)

estimator.options.default_shots = 100
estimator.options.resilience_level = 0
estimator.options.resilience.zne_mitigation = True

### Kasus khusus
#### Tingkat resiliensi (hanya Estimator)
Tingkat resiliensi sebenarnya bukan opsi yang langsung memengaruhi kueri primitif, tetapi menentukan sekumpulan opsi terkurasi dasar untuk dibangun. Secara umum, level 0 menonaktifkan semua mitigasi error, level 1 mengaktifkan opsi untuk mitigasi error pengukuran, dan level 2 mengaktifkan opsi untuk mitigasi error gate dan pengukuran.

Opsi apa pun yang kamu tentukan secara manual selain tingkat resiliensi diterapkan di atas sekumpulan opsi dasar yang didefinisikan oleh tingkat resiliensi. Jadi, pada prinsipnya, kamu bisa mengatur tingkat resiliensi ke 1, tetapi kemudian menonaktifkan mitigasi pengukuran, meskipun ini tidak disarankan.

Dalam contoh berikut, mengatur tingkat resiliensi ke 0 awalnya menonaktifkan `zne_mitigation`, tetapi `estimator.options.resilience.zne_mitigation = True` menimpa pengaturan yang relevan dari `estimator.options.resilience_level = 0`.

In [6]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)


# Setting shots during primitive initialization
sampler = Sampler(mode=backend, options={"default_shots": 4096})

# Setting options after primitive initialization
# This uses auto-complete.
sampler.options.default_shots = 2000

# This does bulk update.  The value for default_shots is overridden if you specify shots with run() or in the PUB.
sampler.options.update(
    default_shots=1024, dynamical_decoupling={"sequence_type": "XpXm"}
)

# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d5k96icjt3vs73ds5t0g', 'sampler')>

#### Shots (hanya Sampler)
Metode `SamplerV2.run` menerima dua argumen: daftar PUB, yang masing-masing bisa menentukan nilai shots khusus PUB, dan argumen kata kunci shots. Nilai shots ini merupakan bagian dari antarmuka eksekusi Sampler, dan independen dari opsi Runtime Sampler. Nilai-nilai ini lebih diprioritaskan daripada nilai yang ditentukan sebagai opsi untuk mematuhi abstraksi Sampler.

Namun, jika `shots` tidak ditentukan oleh PUB mana pun atau dalam argumen kata kunci run (atau jika semuanya `None`), maka nilai shots dari opsi digunakan, terutama `default_shots`.

Ringkasnya, ini adalah urutan prioritas untuk menentukan shots dalam Sampler, untuk PUB tertentu:

1. Jika PUB menentukan shots, gunakan nilai tersebut.
2. Jika argumen kata kunci `shots` ditentukan dalam `run`, gunakan nilai tersebut.
3. Jika `num_randomizations` dan `shots_per_randomization` ditentukan sebagai opsi `twirling`, shots adalah hasil kali nilai-nilai tersebut.
3. Jika `sampler.options.default_shots` ditentukan, gunakan nilai tersebut.

Jadi, jika shots ditentukan di semua tempat yang memungkinkan, yang dengan prioritas tertinggi (shots yang ditentukan dalam PUB) yang digunakan.

#### Precision (hanya Estimator)
Precision analog dengan shots, seperti yang dijelaskan di bagian sebelumnya, kecuali bahwa opsi Estimator berisi `default_shots` dan `default_precision`. Selain itu, karena gate-twirling diaktifkan secara default, hasil kali `num_randomizations` dan `shots_per_randomization` lebih diprioritaskan daripada kedua opsi tersebut.

Secara khusus, untuk PUB Estimator tertentu:

1. Jika PUB menentukan precision, gunakan nilai tersebut.
2. Jika argumen kata kunci precision ditentukan dalam `run`, gunakan nilai tersebut.
2. Jika `num_randomizations` dan `shots_per_randomization` ditentukan sebagai [opsi `twirling`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options) (diaktifkan secara default), gunakan hasil kalinya untuk mengontrol jumlah data.
3. Jika `estimator.options.default_shots` ditentukan, gunakan nilai tersebut untuk mengontrol jumlah data.
4. Jika `estimator.options.default_precision` ditentukan, gunakan nilai tersebut.

Misalnya, jika precision ditentukan di keempat tempat, yang dengan prioritas tertinggi (precision yang ditentukan dalam PUB) yang digunakan.

> **Note:** Precision berbanding terbalik dengan penggunaan. Artinya, semakin rendah precision, semakin banyak waktu QPU yang dibutuhkan untuk menjalankannya.

## Opsi yang umum digunakan
Ada banyak opsi yang tersedia, tetapi berikut ini yang paling umum digunakan:

<span id="shots"></span>
### Shots
Untuk beberapa algoritma, mengatur jumlah shots tertentu adalah bagian inti dari rutinitas mereka. Shots (atau precision) bisa ditentukan di beberapa tempat. Prioritasnya adalah sebagai berikut:

Untuk PUB Sampler mana pun:

1. Shots bernilai integer yang terkandung dalam PUB
2. Nilai `run(...,shots=val)`
3. Nilai `options.default_shots`

Untuk PUB Estimator mana pun:

1. Precision bernilai float yang terkandung dalam PUB
2. Nilai `run(...,precision=val)`
3. Nilai `options.default_shots`
4. Nilai `options.default_precision`

Contoh:

In [7]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(mode=backend)

estimator.options.max_execution_time = 2500

<span id="no-error-mitigation"></span>
## Turn off all error mitigation and error suppression

You can turn off all error mitigation and suppression if you are, for example, doing research on your own mitigation techniques. To accomplish this, for EstimatorV2, set `resilience_level = 0`. For SamplerV2, no changes are necessary because no error mitigation or suppression options are enabled by default.

Example:

Turn off all error mitigation and suppression in Estimator.

In [8]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator, QiskitRuntimeService

# Define the service.  This allows you to access IBM QPU.
service = QiskitRuntimeService()

# Get a backend
backend = service.least_busy(operational=True, simulator=False)

# Define Estimator
estimator = Estimator(backend)

options = estimator.options

# Turn off all error mitigation and suppression
options.resilience_level = 0

### Waktu eksekusi maksimum
Waktu eksekusi maksimum (`max_execution_time`) membatasi berapa lama suatu job bisa berjalan. Jika suatu job melebihi batas waktu ini, job tersebut dibatalkan secara paksa. Nilai ini berlaku untuk job tunggal, baik dijalankan dalam mode job, Session, maupun batch.

Nilainya diatur dalam detik, berdasarkan waktu kuantum (bukan wall clock time), yaitu jumlah waktu QPU yang didedikasikan untuk memproses job kamu. Nilai ini diabaikan saat menggunakan mode pengujian lokal karena mode tersebut tidak menggunakan waktu kuantum.